# Qwen2.5-7B + LoRA — Platrixa Specialist Finance Model

P5a training with real FYJC accounting data.

Based on Unsloth's Qwen2.5 Alpaca notebook. Modified to train on
`specialist_clean_training.jsonl` (46 validated accounting examples).

**Runtime > Run all** on a free Tesla T4 Google Colab instance.

## Installation

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

## Unsloth — Model Loading

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

## LoRA Configuration

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

## Data Prep — Real Platrixa Accounting Data

Replaces the Alpaca/Fibonacci demo with `specialist_clean_training.jsonl`.

**First: upload `specialist_clean_training.jsonl` to Colab**
(`Files` panel → upload → select the file from this repo's `training_data/` folder)

In [ ]:
# ============================================================
# P5a — Load real Platrixa accounting training data
# ============================================================

import json
from datasets import load_dataset

# --- 1. Load the real JSONL ---
# After uploading specialist_clean_training.jsonl to Colab,
# it will appear at this path:
TRAINING_JSONL = "/content/specialist_clean_training.jsonl"

dataset = load_dataset("json", data_files=TRAINING_JSONL, split="train")

# --- 2. Validate ---
print(f"File:          {TRAINING_JSONL}")
print(f"Record count:  {len(dataset)}")
print(f"Columns:       {dataset.column_names}")
print(f"Output type:   {type(dataset[0]['output']).__name__}")

# Check for empty fields
empty_inst = sum(1 for r in dataset if not str(r.get("","")).strip())
empty_in   = sum(1 for r in dataset if not str(r.get("input","")).strip())
empty_out  = sum(1 for r in dataset if not str(r.get("output","")).strip())
print(f"Empty instruction: {empty_inst}/{len(dataset)}")
print(f"Empty input:       {empty_in}/{len(dataset)}")
print(f"Empty output:      {empty_out}/{len(dataset)}")

# Verify output is valid JSON
bad_json = 0
for r in dataset:
    try:
        json.loads(r["output"])
    except:
        bad_json += 1
print(f"Invalid JSON outputs: {bad_json}/{len(dataset)}")

# --- 3. Show one sample (NOT the whole dataset) ---
print("\n=== Sample record ===")
s = dataset[0]
print(f"Instruction: {s['instruction'][:120]}...")
print(f"Input:       {s['input']}")
out = json.loads(s["output"])
print(f"Output keys: {list(out.keys())}")
print(f"  transaction_type: {out.get('transaction_type')}")
print(f"  parties:          {out.get('parties')}")
print(f"  payment_method:   {out.get('payment_method')}")

# --- 4. Confirm NO demo/Fibonacci data ---
print("\n=== Confirmation ===")
has_fib = any("fibonacci" in str(r).lower() or "alpaca" in str(r).lower() for r in dataset)
print(f"Fibonacci/Alpaca records: {'YES - ERROR' if has_fib else 'NONE - CORRECT'}")
print(f"Real Platrixa accounting data: {len(dataset)} records ready")

## Formatting Function — Accounting Prompt

Converts each JSONL record into the Alpaca prompt format expected by SFTTrainer.

In [ ]:
# ============================================================
# P5a — Accounting prompt formatter
# ============================================================

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    """Convert Platrixa accounting records into Alpaca prompt format.

    Input record: {instruction, input, output}
    - instruction: task description for the model
    - input: raw student accounting text
    - output: JSON string with structured interpretation

    The output JSON is preserved exactly as the training target.
    """
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input_text, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts }

# Apply formatting
dataset = dataset.map(formatting_prompts_func, batched = True)

# --- Validate formatted output ---
print(f"Formatted records: {len(dataset)}")
print(f"Columns: {dataset.column_names}")

# Show one formatted prompt (truncated)
print("\n=== Sample formatted prompt ===")
sample_text = dataset[0]["text"]
lines = sample_text.split("\n")
for line in lines[:15]:
    print(f"  {line}")
if len(lines) > 15:
    print(f"  ... ({len(lines) - 15} more lines)")

# Confirm no Fibonacci
any_fib = any("fibonacci" in r["text"].lower() for r in dataset.select(range(min(5, len(dataset)))))
print(f"\nFibonacci in formatted data: {'YES - ERROR' if any_fib else 'NO - CORRECT'}")
print(f"Training examples ready: {len(dataset)}")

## Train the Model

Uses the real Platrixa accounting dataset. Set `max_steps` higher for a full run.

In [ ]:
from trl import SFTConfig, SFTTrainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = False,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # num_train_epochs = 1,  # Set this for a full training run
        max_steps = 60,         # Increase for real training (e.g. 200-500)
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

In [ ]:
trainer_stats = trainer.train()

## Inference — Test the Trained Model

Try a real accounting question instead of Fibonacci.

In [ ]:
FastLanguageModel.for_inference(model)

inputs = tokenizer(
[
    alpaca_prompt.format(
        "Parse the student's accounting language into a grounded structured interpretation. Do not invent missing information.",
        "Purchased goods from Raj for Rs.20000 by cheque.",
        "",  # output - leave blank for generation
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 256, use_cache = True)
tokenizer.batch_decode(outputs)

In [ ]:
# TextStreamer for continuous inference
FastLanguageModel.for_inference(model)

inputs = tokenizer(
[
    alpaca_prompt.format(
        "Parse the student's accounting language into a grounded structured interpretation. Do not invent missing information.",
        "Paid electricity bill Rs.2800.",
        "",
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 256)

In [ ]:
# Test with an ambiguous case
FastLanguageModel.for_inference(model)

inputs = tokenizer(
[
    alpaca_prompt.format(
        "Parse the student's accounting language into a grounded structured interpretation. Do not invent missing information.",
        "Purchased goods from Raj for Rs.5000.",
        "",
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 256, use_cache = True)
tokenizer.batch_decode(outputs)

## Save the Model

In [ ]:
model.save_pretrained("qwen_platrixa_lora")
tokenizer.save_pretrained("qwen_platrixa_lora")
# model.push_to_hub("your_name/qwen_platrixa_lora", token = "YOUR_HF_TOKEN")
# tokenizer.push_to_hub("your_name/qwen_platrixa_lora", token = "YOUR_HF_TOKEN")

## Load Saved LoRA for Later Inference

In [ ]:
if False:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "qwen_platrixa_lora",
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model)

inputs = tokenizer(
[
    alpaca_prompt.format(
        "Parse the student's accounting language into a grounded structured interpretation.",
        "Sold goods to Amit for Rs.15000 on credit.",
        "",
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 256)

## Save to GGUF / llama.cpp

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("qwen_platrixa_finetune", tokenizer,)

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("qwen_platrixa_finetune", tokenizer, quantization_method = "q4_k_m")

---

**P5a Note:** This notebook trains on 46 validated Platrixa accounting examples.

The model learns to parse student accounting language into structured JSON interpretations.

The Truth Kernel remains separate — it handles all accounting computation, verification, and journal generation.